## 清洗爬取的数据

In [ ]:
# -*- coding: utf-8 -*-
"""
把 org_data.json 和 usr_data.json 合并
用法
"""

import json
import sys
from pathlib import Path

def load_json(path: Path) -> dict:
    """读取 json 文件并转为 dict"""
    try:
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"[ERROR] 文件不存在: {path}")
        sys.exit(1)
    except json.JSONDecodeError as e:
        print(f"[ERROR] 解析 JSON 失败: {path}\n{e}")
        sys.exit(1)

def save_json(data: dict, path: Path):
    """把 dict 保存为 json 文件"""
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"[OK] 已写入 {path}（共 {len(data)} 条记录）")

def merge_json(org_file: Path, usr_file: Path, out_file: Path):
    org_dict = load_json(org_file)
    usr_dict = load_json(usr_file)

    # 合并，后者覆盖前者
    merged = {**org_dict, **usr_dict}

    save_json(merged, out_file)

if __name__ == "__main__":
    # 默认文件名
    org_fp = Path("raw/author_metadata/org_data.json")
    usr_fp = Path("raw/author_metadata/usr_data.json")
    out_fp = Path("preprocessed/author_metadata_tmp.json")
    merge_json(org_fp, usr_fp, out_fp)


In [ ]:
import json

def clean_author_metadata(file_path, write_path):
    """
    清洗 author_metadata.json 中的 photo 字段，为不以 'https' 开头的值补充前缀 'https://huggingface.co/'。
    Args:
        file_path (str): 文件路径
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        author_data = json.load(f)

    # 遍历所有作者信息并清洗 photo 字段
    for author, details in author_data.items():
        photo_url = details.get('photo', '')
        if not photo_url.startswith('https'):
            # 补充前缀
            author_data[author]['photo'] = f"https://huggingface.co{photo_url}"

    # 写入清洗后的数据回到文件中
    with open(write_path, 'w', encoding='utf-8') as f:
        json.dump(author_data, f, ensure_ascii=False, indent=4)
        print(f"Data cleaning complete. Updated data saved to {write_path}")

# 示例：指定 author_metadata.json 的路径
file_path = "preprocessed/author_metadata_tmp.json"
write_path = "preprocessed/author_metadata.json"
clean_author_metadata(file_path,write_path)
# 删除临时文件
Path("preprocessed/author_metadata_tmp.json").unlink(missing_ok=True)


In [ ]:
import json
import os

def clean_model_metadata(metadata_file, tree_file, output_file):
    """
    清洗 model_metadata.json，通过填补衍生模型缺失的 'pipeline_tag' 属性。

    Args:
        metadata_file (str): model_metadata.json 的路径。
        tree_file (str): model_tree_raw.json 的路径。
        output_file (str): 清洗后数据的保存路径。
    """

    # 检查文件是否存在
    if not os.path.exists(metadata_file):
        print(f"错误：文件 {metadata_file} 不存在。")
        return
    if not os.path.exists(tree_file):
        print(f"错误：文件 {tree_file} 不存在。")
        return

    # 加载 model_metadata.json
    with open(metadata_file, 'r', encoding='utf-8') as f:
        try:
            model_metadata = json.load(f)
            print(f"成功加载 {metadata_file}。")
        except json.JSONDecodeError as e:
            print(f"错误：无法解析 {metadata_file}。错误信息：{e}")
            return

    # 加载 model_tree_raw.json
    with open(tree_file, 'r', encoding='utf-8') as f:
        try:
            model_tree_raw = json.load(f)
            print(f"成功加载 {tree_file}。")
        except json.JSONDecodeError as e:
            print(f"错误：无法解析 {tree_file}。错误信息：{e}")
            return

    # 记录缺失 'pipeline_tag' 的衍生模型
    missing_pipeline_tags = []

    # 遍历每个基模型及其衍生模型
    for base_model, dependencies in model_tree_raw.items():
        # 获取基模型的 'pipeline_tag'，默认为 None
        base_pipeline_tag = model_metadata.get(base_model, {}).get('pipeline_tag')

        for dep_type, derived_models in dependencies.items():
            for derived_model in derived_models:
                # 检查衍生模型是否存在于 model_metadata.json 中
                if derived_model in model_metadata:
                    derived_pipeline_tag = model_metadata[derived_model].get('pipeline_tag')

                    if derived_pipeline_tag is None:
                        if base_pipeline_tag is not None:
                            # 填补衍生模型的 'pipeline_tag' 为基模型的值
                            model_metadata[derived_model]['pipeline_tag'] = base_pipeline_tag
                            print(f"更新模型 '{derived_model}' 的 'pipeline_tag' 为基模型 '{base_model}' 的值：'{base_pipeline_tag}'。")
                        else:
                            # 如果基模型的 'pipeline_tag' 也是 None，设置为 'nan'
                            model_metadata[derived_model]['pipeline_tag'] = 'nan'
                            missing_pipeline_tags.append(derived_model)
                            print(f"更新模型 '{derived_model}' 的 'pipeline_tag' 为 'nan'。")

    # 如果存在缺失 'pipeline_tag' 的模型，记录日志
    if missing_pipeline_tags:
        print("\n警告：以下衍生模型的 'pipeline_tag' 被设置为 'nan'，因为它们的基模型的 'pipeline_tag' 也是缺失的：")
        for model in missing_pipeline_tags:
            print(f"- {model}")

    # 保存清洗后的数据到新的 JSON 文件
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(model_metadata, f, indent=4, ensure_ascii=False)
        print(f"\n清洗后的数据已保存到 {output_file}。")

    print("\n数据清洗完成。")

if __name__ == "__main__":
    # 定义文件路径
    metadata_file = 'raw/model_metadata/model_metadata.json'
    tree_file = 'raw/model_tree/model_tree_raw.json'
    output_file = 'preprocessed/model_metadata.json'

    # 调用清洗函数
    clean_model_metadata(metadata_file, tree_file, output_file)


In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
串行下载作者头像，并显示 tqdm 进度条。
pip install tqdm
"""

import os
import json
import requests
from urllib.parse import quote
from tqdm.auto import tqdm   # << 新增

def download_image(image_url, save_path):
    """下载图片并保存到指定路径。成功→True，失败→False"""
    try:
        if os.path.exists(save_path):
            # 已存在可直接返回 True；如需提示可取消注释下一行
            # print(f"图片 {save_path} 已存在，跳过下载。")
            return True

        resp = requests.get(image_url, timeout=30)
        resp.raise_for_status()
        with open(save_path, "wb") as f:
            f.write(resp.content)
        # print(f"已保存 {save_path}")
        return True
    except (requests.RequestException, OSError) as e:
        print(f"❌ 下载失败 {image_url} -> {e}")
        return False

def main():
    # ---------- 准备路径 ----------
    save_dir   = Path("avatars")
    error_log  = save_dir / "error_downloading_avatars.txt"
    save_dir.mkdir(parents=True, exist_ok=True)

    # ---------- 读取 JSON ----------
    with open("preprocessed/author_metadata.json", "r", encoding="utf-8") as f:
        author_data = json.load(f)

    # 提取含 photo 的条目
    items = [(k, v["photo"]) for k, v in author_data.items() if v.get("photo")]

    # ---------- 主循环 ----------
    error_keys = []
    for key, url in tqdm(items, desc="Downloading", unit="img"):
        img_name  = quote(url.split("/")[-1])
        save_path = save_dir / img_name
        ok = download_image(url, save_path)
        if not ok:
            error_keys.append(key)

    # ---------- 记录失败 ----------
    if error_keys:
        error_log.write_text("\n".join(error_keys), encoding="utf-8")
        print(f"\n⚠ 共 {len(error_keys)} 张下载失败，已记录到 {error_log}")
    else:
        if error_log.exists():
            error_log.unlink()
        print("\n✅ 全部头像下载完成，无错误")

if __name__ == "__main__":
    from pathlib import Path
    main()


C:\Users\zzsyp\AppData\Roaming\Python\Python39\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Downloading:  18%|█▊        | 6582/37414 [00:09<00:25, 1188.14img/s]

❌ 下载失败 https://cdn-avatars.huggingface.co/v1/production/uploads/no-auth/DzPsGdtQKy09YgO2SEtCd.png -> 500 Server Error: Internal Server Error for url: https://cdn-avatars.huggingface.co/v1/production/uploads/no-auth/DzPsGdtQKy09YgO2SEtCd.png


Downloading:  47%|████▋     | 17631/37414 [00:14<00:15, 1250.12img/s]

❌ 下载失败 https://cdn-avatars.huggingface.co/v1/production/uploads/no-auth/4GvZK9V4q4Pcg5CLFLQor.png -> 500 Server Error: Internal Server Error for url: https://cdn-avatars.huggingface.co/v1/production/uploads/no-auth/4GvZK9V4q4Pcg5CLFLQor.png


Downloading:  57%|█████▋    | 21462/37414 [00:15<00:10, 1487.86img/s]

...

❌ 下载失败 https://cdn-avatars.huggingface.co/v1/production/uploads/no-auth/I1OAjJhLHJ0RsONPxJdME.png -> 500 Server Error: Internal Server Error for url: https://cdn-avatars.huggingface.co/v1/production/uploads/no-auth/I1OAjJhLHJ0RsONPxJdME.png


Downloading:  91%|█████████▏| 34155/37414 [00:20<00:02, 1537.88img/s]

❌ 下载失败 https://cdn-avatars.huggingface.co/v1/production/uploads/no-auth/HJSfr2yeRnFD7VMPTGUFt.png -> 500 Server Error: Internal Server Error for url: https://cdn-avatars.huggingface.co/v1/production/uploads/no-auth/HJSfr2yeRnFD7VMPTGUFt.png


Downloading: 100%|██████████| 37414/37414 [01:24<00:00, 443.17img/s] 


⚠ 共 5 张下载失败，已记录到 avatars\error_downloading_avatars.txt


In [2]:
import json
from pathlib import Path
from urllib.parse import urlparse

json_path = Path("preprocessed/author_metadata.json")

# 1. 加载 JSON 文件
with json_path.open("r", encoding="utf-8") as f:
    data = json.load(f)

# 2. 替换 photo 字段
for author_id, info in data.items():
    old_photo = info.get("photo", "")
    if old_photo and isinstance(old_photo, str):
        # 从 URL 中提取文件名
        filename = Path(urlparse(old_photo).path).name
        # 替换前缀
        info["photo"] = f"/static/avatars/{filename}"

# 3. 写回原文件（或另存为新文件）
with json_path.open("w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print("✅ 已替换所有 photo 字段的前缀为 /static/avatars/")


✅ 已替换所有 photo 字段的前缀为 /static/avatars/


In [ ]:
import shutil
from pathlib import Path

# 定义源文件路径列表
file_list = [
    "raw/space_metadata/spaces_metadata.json",
    "raw/model_tree/model_tree_raw.json",
    "raw/basemodel/basemodels_top1000_likes.txt",
]

# 目标目录
target_dir = Path("preprocessed")
target_dir.mkdir(parents=True, exist_ok=True)  # 如果目标目录不存在则创建

# 逐个复制文件
for file_path in file_list:
    src = Path(file_path)
    dst = target_dir / src.name  # 保留原文件名
    if src.exists():
        shutil.copy2(src, dst)  # 使用copy2以保留元数据（时间戳等）
        print(f"✅ 已复制: {src} → {dst}")
    else:
        print(f"⚠️ 文件未找到: {src}")


✅ 已移动: raw\space_metadata\spaces_metadata.json → preprocessed\spaces_metadata.json
✅ 已移动: raw\model_tree\model_tree_raw.json → preprocessed\model_tree_raw.json
✅ 已移动: raw\basemodel\basemodels_top1000_likes.txt → preprocessed\basemodels_top1000_likes.txt
